In [16]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PARQUET_PATH = "local_parquet/chess_moves_raw_parquet"

spark = SparkSession.builder.getOrCreate()
df = spark.read.parquet(PARQUET_PATH)

df.printSchema()
print("rows =", df.count())
df.select("final_result_class").groupBy("final_result_class").count().show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/09 13:02:20 WARN Utils: Your hostname, MacBook-Pro-Bulat.local, resolves to a loopback address: 127.0.0.1; using 192.168.31.14 instead (on interface en0)
26/05/09 13:02:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/09 13:02:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

root
 |-- game_uuid: string (nullable = true)
 |-- game_url: string (nullable = true)
 |-- game_year: integer (nullable = true)
 |-- game_month: integer (nullable = true)
 |-- game_end_timestamp: integer (nullable = true)
 |-- game_end_datetime_utc: timestamp (nullable = true)
 |-- rated: boolean (nullable = true)
 |-- rules: string (nullable = true)
 |-- time_class: string (nullable = true)
 |-- time_control_raw: string (nullable = true)
 |-- time_control_base_seconds: integer (nullable = true)
 |-- time_control_increment_seconds: integer (nullable = true)
 |-- eco_url: string (nullable = true)
 |-- eco_code: string (nullable = true)
 |-- opening_name: string (nullable = true)
 |-- white_username: string (nullable = true)
 |-- black_username: string (nullable = true)
 |-- white_rating: integer (nullable = true)
 |-- black_rating: integer (nullable = true)
 |-- rating_diff: integer (nullable = true)
 |-- white_accuracy: double (nullable = true)
 |-- black_accuracy: double (nullable = t

rows = 660785


[Stage 5:>                                                        (0 + 12) / 12]

+------------------+------+
|final_result_class| count|
+------------------+------+
|         white_win|299117|
|         black_win|307065|
|              draw| 54603|
+------------------+------+



In [18]:
def check_nulls(df):
    null_stats = (
        df.select([
            F.count(F.when(F.col(c).isNull(), 1)).alias(c)
            for c in df.columns
        ])
        .toPandas()
        .T
        .reset_index()
    )
    
    null_stats.columns = ["column", "null_count"]
    null_stats["null_pct"] = (null_stats["null_count"] / df.count()) * 100
    
    null_stats = null_stats[null_stats["null_count"] > 0].sort_values("null_pct", ascending=False)
    
    return null_stats

check_nulls(df)

26/05/09 13:02:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

,column,null_count,null_pct
35,promotion_piece,658229,99.613187
20,white_accuracy,496570,75.148498
21,black_accuracy,496570,75.148498
56,avg_time_spent_per_move_so_far,17790,2.692252
51,black_clock_seconds_after,8889,1.345218
10,time_control_base_seconds,435,0.065831
11,time_control_increment_seconds,435,0.065831
55,clock_remaining_pct,435,0.065831
53,side_to_move_clock_before,18,0.002724
54,time_spent_on_move_seconds,18,0.002724


In [19]:
# filling in "avg_time_spent_per_move_so_far"

df = df.withColumn(
    "avg_time_spent_missing",
    F.col("avg_time_spent_per_move_so_far").isNull().cast("int")
)

med = (df
       .where(F.col("avg_time_spent_per_move_so_far").isNotNull())
       .groupBy("time_class")
       .agg(F.expr("percentile_approx(avg_time_spent_per_move_so_far, 0.5)").alias("med_avg_time"))
      )

df = (df.join(med, on="time_class", how="left")
        .withColumn(
            "avg_time_spent_per_move_so_far",
            F.coalesce(F.col("avg_time_spent_per_move_so_far"), F.col("med_avg_time"))
        )
        .drop("med_avg_time")
     )

Let's inspect *time_control_raw* values when *time_control_base_seconds* is Null

In [20]:
# find "time_control_raw" distribution when time_control_base_seconds is Null

df.filter(F.col("time_control_base_seconds").isNull()) \
  .select("time_control_raw","time_class","rated") \
  .groupBy("time_control_raw","time_class").count().show(50, False)

[Stage 17:=========>                                              (2 + 10) / 12]

+----------------+----------+-----+
|time_control_raw|time_class|count|
+----------------+----------+-----+
|1/604800        |daily     |433  |
|1/86400         |daily     |2    |
+----------------+----------+-----+



In [21]:
df.groupBy("time_class").count().show()

+----------+------+
|time_class| count|
+----------+------+
|     blitz|302795|
|    bullet|259819|
|     rapid| 97736|
|     daily|   435|
+----------+------+



> We see that values of format A/B (for instance **1/604800**) belong to daily group of *time_class* feature and their quantity is 435. Also notice number of rows with *time_class* = **daily** is 435. So we can conclude that *time_control_raw* values are following the format of A/B when *time_class* is **daily**.

> Remember that *time_control_base_seconds* and *time_control_increment_seconds* were parsed from *time_control_raw* feature. It's easy to guess that A is time increment (in seconds) and B is time limit (for example 604800 seconds are 7 days).

In [22]:
tc = F.trim(F.col("time_control_raw"))

daily_slash_vals = ["1/86400", "1/604800"]

df = df.withColumn(
    "time_control_base_seconds",
    F.when(
        F.col("time_control_base_seconds").isNull() & (tc == F.lit("1/86400")),
        F.lit(86400)
    ).when(
        F.col("time_control_base_seconds").isNull() & (tc == F.lit("1/604800")),
        F.lit(604800)
    ).otherwise(F.col("time_control_base_seconds"))
)

df = df.withColumn(
    "time_control_increment_seconds",
    F.when(
        F.col("time_control_increment_seconds").isNull() &
        (tc.isin(daily_slash_vals)),
        F.lit(1)
    ).otherwise(F.col("time_control_increment_seconds"))
)

df = df.withColumn(
    "clock_remaining_pct",
    F.when(
        F.col("clock_remaining_pct").isNull() &
        (tc.isin(daily_slash_vals)) &
        F.col("side_to_move_clock_before").isNotNull() &
        (F.col("time_control_base_seconds") > 0),
        F.col("side_to_move_clock_before").cast("double") /
        F.col("time_control_base_seconds").cast("double")
    ).when(
        F.col("clock_remaining_pct").isNull() &
        (tc.isin(daily_slash_vals)) &
        (F.col("ply_index").isin([1, 2])),
        F.lit(1.0)
    ).otherwise(F.col("clock_remaining_pct"))
)

In [23]:
check_nulls(df)

,column,null_count,null_pct
35,promotion_piece,658229,99.613187
20,white_accuracy,496570,75.148498
21,black_accuracy,496570,75.148498
51,black_clock_seconds_after,8889,1.345218
53,side_to_move_clock_before,18,0.002724
54,time_spent_on_move_seconds,18,0.002724
57,is_in_time_trouble_30s,18,0.002724


Let's also inspect values in *side_to_move_clock_before* and *is_in_time_trouble_30s*.

In [24]:
df.filter(F.col("side_to_move_clock_before").isNull()) \
  .select("time_control_raw", "side_to_move_clock_before", "is_in_time_trouble_30s", "clock_remaining_pct", "time_class", "ply_index").show()

+----------------+-------------------------+----------------------+-------------------+----------+---------+
|time_control_raw|side_to_move_clock_before|is_in_time_trouble_30s|clock_remaining_pct|time_class|ply_index|
+----------------+-------------------------+----------------------+-------------------+----------+---------+
|        1/604800|                     NULL|                  NULL|                1.0|     daily|        1|
|        1/604800|                     NULL|                  NULL|                1.0|     daily|        2|
|         1/86400|                     NULL|                  NULL|                1.0|     daily|        1|
|         1/86400|                     NULL|                  NULL|                1.0|     daily|        2|
|        1/604800|                     NULL|                  NULL|                1.0|     daily|        1|
|        1/604800|                     NULL|                  NULL|                1.0|     daily|        2|
|        1/604800| 

From this example, it becomes clear that side_to_move_clock_before must be filled with values of *time_control_base_seconds* because *ply_index* values indicate that these rows represent first move of opponents in the game. Also obvious that *is_in_time_trouble_30s* must be False for all these rows.

In [25]:
daily_slash = ["1/86400", "1/604800"]
tc = F.trim(F.col("time_control_raw"))
mask = tc.isin(daily_slash)

base_d = F.col("time_control_base_seconds").cast("double")
side_d = F.col("side_to_move_clock_before").cast("double")

side_filled = F.when(mask,
                     F.coalesce(side_d, base_d)
                    ).otherwise(side_d)

df = df.withColumn("side_to_move_clock_before", side_filled)

df = df.withColumn(
    "is_in_time_trouble_30s",
    F.when(
        mask & F.col("is_in_time_trouble_30s").isNull(),
        (F.col("side_to_move_clock_before") < F.lit(30)).cast("boolean")
    ).otherwise(F.col("is_in_time_trouble_30s"))
)

print("NULL side_to_move_clock_before (daily slash):",
      df.filter(mask & F.col("side_to_move_clock_before").isNull()).count())

print("NULL is_in_time_trouble_30s (daily slash):",
      df.filter(mask & F.col("is_in_time_trouble_30s").isNull()).count())

NULL side_to_move_clock_before (daily slash): 0


[Stage 38:>                                                       (0 + 12) / 12]

NULL is_in_time_trouble_30s (daily slash): 0


In [26]:
check_nulls(df)

,column,null_count,null_pct
35,promotion_piece,658229,99.613187
20,white_accuracy,496570,75.148498
21,black_accuracy,496570,75.148498
51,black_clock_seconds_after,8889,1.345218
54,time_spent_on_move_seconds,18,0.002724


In [27]:
features = [
    "game_uuid",
    "rated",
    "rules",
    "time_class",
    "time_control_base_seconds",
    "time_control_increment_seconds",
    # "eco_code",
    "white_rating",
    "black_rating",
    "rating_diff",
    "ply_index",
    "fullmove_number",
    "side_to_move",
    "is_capture",
    "is_check",
    "is_checkmate",
    "is_castling",
    "is_promotion",
    # "promotion_piece",
    # "from_square",
    # "to_square",
    "piece_moved",
    # "en_passant_square_before",
    "halfmove_clock_before",
    "legal_moves_count_before",
    "material_white_before",
    "material_black_before",
    "material_diff_before",
    "side_to_move_clock_before",
    "clock_remaining_pct",
    "avg_time_spent_per_move_so_far",
    "is_in_time_trouble_30s",
    "white_doubled_pawns_before",
    "black_doubled_pawns_before",
    "doubled_pawns_diff_before",
    "isolated_pawns_diff_before",
    "passed_pawns_diff_before",
    "pawn_shield_diff_before",
    "king_tropism_diff_before",
    "white_can_castle_kingside_before",
    "white_can_castle_queenside_before",
    "black_can_castle_kingside_before",
    "black_can_castle_queenside_before",
    "in_check_before",
    "is_opening_phase",
    "is_middlegame_phase",
    "is_endgame_phase",
    "total_piece_count_before",
    "non_pawn_material_white_before",
    "bishops_pair_white_before",
    "bishops_pair_black_before",
    "final_result_class"
]

df_subset = df.select(*features)

In [28]:
check_nulls(df_subset)

,column,null_count,null_pct


In [29]:
df_subset.printSchema()

root
 |-- game_uuid: string (nullable = true)
 |-- rated: boolean (nullable = true)
 |-- rules: string (nullable = true)
 |-- time_class: string (nullable = true)
 |-- time_control_base_seconds: integer (nullable = true)
 |-- time_control_increment_seconds: integer (nullable = true)
 |-- white_rating: integer (nullable = true)
 |-- black_rating: integer (nullable = true)
 |-- rating_diff: integer (nullable = true)
 |-- ply_index: integer (nullable = true)
 |-- fullmove_number: integer (nullable = true)
 |-- side_to_move: string (nullable = true)
 |-- is_capture: boolean (nullable = true)
 |-- is_check: boolean (nullable = true)
 |-- is_checkmate: boolean (nullable = true)
 |-- is_castling: boolean (nullable = true)
 |-- is_promotion: boolean (nullable = true)
 |-- piece_moved: string (nullable = true)
 |-- halfmove_clock_before: integer (nullable = true)
 |-- legal_moves_count_before: integer (nullable = true)
 |-- material_white_before: integer (nullable = true)
 |-- material_black_be

In [30]:
df_subset.limit(10).toPandas()

,game_uuid,rated,rules,time_class,time_control_base_seconds,time_control_increment_seconds,white_rating,black_rating,rating_diff,ply_index,...,black_can_castle_queenside_before,in_check_before,is_opening_phase,is_middlegame_phase,is_endgame_phase,total_piece_count_before,non_pawn_material_white_before,bishops_pair_white_before,bishops_pair_black_before,final_result_class
0,a6a280d8-cfa0-11f0-b1f3-5623b701000f,True,chess,bullet,60,0,1825,1830,-5,52,...,False,False,False,True,False,20,13,False,False,black_win
1,174b600d-de79-11f0-9c69-fb411201000f,True,chess,blitz,180,0,1571,1609,-38,77,...,False,False,False,True,False,15,11,False,False,black_win
2,a6a280d8-cfa0-11f0-b1f3-5623b701000f,True,chess,bullet,60,0,1825,1830,-5,53,...,False,False,False,True,False,20,13,False,False,black_win
3,174b600d-de79-11f0-9c69-fb411201000f,True,chess,blitz,180,0,1571,1609,-38,78,...,False,False,False,True,False,15,11,False,False,black_win
4,a6a280d8-cfa0-11f0-b1f3-5623b701000f,True,chess,bullet,60,0,1825,1830,-5,54,...,False,False,False,True,False,20,13,False,False,black_win
5,174b600d-de79-11f0-9c69-fb411201000f,True,chess,blitz,180,0,1571,1609,-38,79,...,False,False,False,True,False,15,11,False,False,black_win
6,a6a280d8-cfa0-11f0-b1f3-5623b701000f,True,chess,bullet,60,0,1825,1830,-5,55,...,False,True,False,True,False,20,13,False,False,black_win
7,174b600d-de79-11f0-9c69-fb411201000f,True,chess,blitz,180,0,1571,1609,-38,80,...,False,False,False,True,False,15,11,False,False,black_win
8,a6a280d8-cfa0-11f0-b1f3-5623b701000f,True,chess,bullet,60,0,1825,1830,-5,56,...,False,False,False,True,False,20,13,False,False,black_win
9,174b600d-de79-11f0-9c69-fb411201000f,True,chess,blitz,180,0,1571,1609,-38,81,...,False,False,False,True,False,15,11,False,False,black_win


In [31]:
from pyspark.sql.types import BooleanType

label_col = "final_result_class"

cat_cols = ["rules", "time_class", "side_to_move", "piece_moved"]

bool_cols = [f.name for f in df_subset.schema.fields if isinstance(f.dataType, BooleanType)]

num_cols = [c for c in df_subset.columns if c not in cat_cols + bool_cols + [label_col] + ["game_uuid"]]

print("cat_cols:", cat_cols, '\n')
print("bool_cols:", bool_cols, '\n')
print("num_cols count:", num_cols)

cat_cols: ['rules', 'time_class', 'side_to_move', 'piece_moved'] 

bool_cols: ['rated', 'is_capture', 'is_check', 'is_checkmate', 'is_castling', 'is_promotion', 'is_in_time_trouble_30s', 'white_can_castle_kingside_before', 'white_can_castle_queenside_before', 'black_can_castle_kingside_before', 'black_can_castle_queenside_before', 'in_check_before', 'is_opening_phase', 'is_middlegame_phase', 'is_endgame_phase', 'bishops_pair_white_before', 'bishops_pair_black_before'] 

num_cols count: ['time_control_base_seconds', 'time_control_increment_seconds', 'white_rating', 'black_rating', 'rating_diff', 'ply_index', 'fullmove_number', 'halfmove_clock_before', 'legal_moves_count_before', 'material_white_before', 'material_black_before', 'material_diff_before', 'side_to_move_clock_before', 'clock_remaining_pct', 'avg_time_spent_per_move_so_far', 'white_doubled_pawns_before', 'black_doubled_pawns_before', 'doubled_pawns_diff_before', 'isolated_pawns_diff_before', 'passed_pawns_diff_before', 'pawn_

In [32]:
# Type conversion (booleans and numeric)
for c in bool_cols:
    df_subset = df_subset.withColumn(c, F.col(c).cast("int").cast("double"))

for c in num_cols:
    df_subset = df_subset.withColumn(c, F.col(c).cast("double"))

In [33]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, Imputer, VectorAssembler

numeric_for_imputer = num_cols + bool_cols

imputed_cols = [c + "_imp" for c in numeric_for_imputer]

label_indexer = StringIndexer(
    inputCol=label_col, outputCol="label", handleInvalid="skip"
)

indexers = []
ohe = []
for c in cat_cols:
    idx_col = c + "_idx"
    ohe_col = c + "_ohe"
    indexers.append(
        StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid="keep")
    )
    ohe.append(
        OneHotEncoder(inputCol=idx_col, outputCol=ohe_col)
    )

imputer = Imputer(
    strategy="median",
    inputCols=numeric_for_imputer,
    outputCols=imputed_cols
)

numeric_assembler = VectorAssembler(
    inputCols=imputed_cols,
    outputCol="numeric_features"
)

ohe_cols = [c + "_ohe" for c in cat_cols]

final_assembler = VectorAssembler(
    inputCols=["numeric_features"] + ohe_cols,
    outputCol="features"
)

preprocess = [label_indexer] + indexers + ohe + [imputer, numeric_assembler, final_assembler]

In [34]:
SEED = 42

games = df_subset.select("game_uuid").distinct().withColumn("r", F.rand(SEED))
train_games = games.filter(F.col("r") < 0.7).select("game_uuid")
test_games  = games.filter(F.col("r") >= 0.7).select("game_uuid")

train_df = df_subset.join(train_games, "game_uuid", "inner")
test_df  = df_subset.join(test_games,  "game_uuid", "inner")

In [35]:
print("Train rows:", train_df.count(), "Test rows:", test_df.count())

[Stage 72:============================>                            (6 + 6) / 12]

Train rows: 469748 Test rows: 191037


# Now it is time for Modeling!

* ### Model 1 - Random Forest

In [37]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# ===== 3) Random Forest + Grid Search (27 combinations) =====
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    seed=SEED
)

paramGrid_rf = (ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100, 200])           # 3
    .addGrid(rf.maxDepth, [5, 10, 20])             # 3
    .addGrid(rf.minInstancesPerNode, [1, 2, 5])    # 3  => 27
    .build()
)

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

# ---- best by Accuracy ----
rf_pipeline = Pipeline(stages=preprocess + [rf])

cv_acc = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=paramGrid_rf,
    evaluator=evaluator_acc,
    numFolds=3,     # 2<k<5
    seed=SEED
)

cvModel_acc = cv_acc.fit(train_df)
best_rf_acc_model = cvModel_acc.bestModel

pred_acc = best_rf_acc_model.transform(test_df)
acc_test = evaluator_acc.evaluate(pred_acc)
f1_test  = evaluator_f1.evaluate(pred_acc)

print("RF BEST by Accuracy -> TEST Accuracy:", acc_test, "TEST F1:", f1_test)
print("Best RF params (by Accuracy):", best_rf_acc_model.stages[-1].extractParamMap())

# ---- best by F1 ----
cv_f1 = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=paramGrid_rf,
    evaluator=evaluator_f1,
    numFolds=3,
    seed=SEED
)

cvModel_f1 = cv_f1.fit(train_df)
best_rf_f1_model = cvModel_f1.bestModel

pred_f1 = best_rf_f1_model.transform(test_df)
acc_test2 = evaluator_acc.evaluate(pred_f1)
f1_test2  = evaluator_f1.evaluate(pred_f1)

print("RF BEST by F1 -> TEST Accuracy:", acc_test2, "TEST F1:", f1_test2)
print("Best RF params (by F1):", best_rf_f1_model.stages[-1].extractParamMap())

26/05/09 13:05:30 WARN DAGScheduler: Broadcasting large task binary with size 1318.5 KiB
26/05/09 13:05:33 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/05/09 13:05:37 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/05/09 13:05:43 WARN DAGScheduler: Broadcasting large task binary with size 1054.2 KiB
26/05/09 13:05:44 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/05/09 13:05:50 ERROR Executor: Exception in task 11.0 in stage 240.0 (TID 1286)
java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.ml.tree.impl.DTStatsAggregator.<init>(DTStatsAggregator.scala:77)
	at org.apache.spark.ml.tree.impl.RandomForest$.$anonfun$findBestSplits$22(RandomForest.scala:681)
	at org.apache.spark.ml.tree.impl.RandomForest$.$anonfun$findBestSplits$22$adapted(RandomForest.scala:677)
	at org.apache.spark.ml.tree.impl.RandomForest$$$Lambda/0x000000012d6f87c8.apply(Unknown Source)
	at scala.Array$.tabulate(Array.scala:443)
	at 

ConnectionRefusedError: [Errno 61] Connection refused

* ### Model 2 - SVM (LinearSVC + OneVsRest)

In [ ]:
from pyspark.ml.classification import LinearSVC, OneVsRest

SEED = 42
LABEL_COL = "final_result_class"

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

# Base classifier
svm = LinearSVC(featuresCol="features", labelCol="label", seed=SEED)

# Multiclass wrapper
ovr = OneVsRest(classifier=svm)

paramGrid_svm = (ParamGridBuilder()
    .addGrid(svm.regParam, [0.001, 0.01, 0.1])     # 3
    .addGrid(svm.maxIter, [20, 50, 100])         # 3
    .addGrid(svm.tol, [1e-4, 1e-3, 1e-2])        # 3 => 27
    .build()
)

svm_pipeline = Pipeline(stages=preprocess.getStages() + [ovr])

# --- best by Accuracy ---
cv_svm_acc = CrossValidator(
    estimator=svm_pipeline,
    estimatorParamMaps=paramGrid_svm,
    evaluator=evaluator_acc,
    numFolds=3,
    seed=SEED,
    parallelism=1
)

cvModel_svm_acc = cv_svm_acc.fit(train_df)
best_svm_acc_model = cvModel_svm_acc.bestModel

pred_svm_acc = best_svm_acc_model.transform(test_df)
acc_svm = evaluator_acc.evaluate(pred_svm_acc)
f1_svm = evaluator_f1.evaluate(pred_svm_acc)

print("SVM BEST by Accuracy -> TEST Accuracy:", acc_svm, "TEST F1:", f1_svm)
print("Best SVM params (by Accuracy):", best_svm_acc_model.stages[-1].extractParamMap())

# --- best by F1 ---
cv_svm_f1 = CrossValidator(
    estimator=svm_pipeline,
    estimatorParamMaps=paramGrid_svm,
    evaluator=evaluator_f1,
    numFolds=3,
    seed=SEED,
    parallelism=1
)

cvModel_svm_f1 = cv_svm_f1.fit(train_df)
best_svm_f1_model = cvModel_svm_f1.bestModel

pred_svm_f1 = best_svm_f1_model.transform(test_df)
acc_svm2 = evaluator_acc.evaluate(pred_svm_f1)
f1_svm2 = evaluator_f1.evaluate(pred_svm_f1)

print("SVM BEST by F1 -> TEST Accuracy:", acc_svm2, "TEST F1:", f1_svm2)
print("Best SVM params (by F1):", best_svm_f1_model.stages[-1].extractParamMap())

* ### Model 3 - Naive Bayes (Multinomial) 

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.feature import MinMaxScaler, QuantileDiscretizer
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

SEED = 42
LABEL_COL = "final_result_class"

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

# scaler -> discretizer -> NB
scaler = MinMaxScaler(inputCol="features", outputCol="scaledFeatures")

discretizer = QuantileDiscretizer(
    inputCol="scaledFeatures",
    outputCol="discFeatures",
    handleInvalid="skip"
)

nb = NaiveBayes(featuresCol="discFeatures", labelCol="label", modelType="multinomial")

# 27 комбинаций: 3 x 3 x 3
paramGrid_nb = (ParamGridBuilder()
    .addGrid(discretizer.numBuckets, [10, 20, 50])                 # 3
    .addGrid(discretizer.relativeError, [0.01, 0.05, 0.1])      # 3
    .addGrid(nb.smoothing, [0.0, 0.5, 1.0])                      # 3 => 27
    .build()
)

nb_pipeline = Pipeline(stages=preprocess.getStages() + [scaler, discretizer, nb])

# --- best by Accuracy ---
cv_nb_acc = CrossValidator(
    estimator=nb_pipeline,
    estimatorParamMaps=paramGrid_nb,
    evaluator=evaluator_acc,
    numFolds=3,
    seed=SEED,
    parallelism=1
)

cvModel_nb_acc = cv_nb_acc.fit(train_df)
best_nb_acc_model = cvModel_nb_acc.bestModel

pred_nb_acc = best_nb_acc_model.transform(test_df)
acc_nb = evaluator_acc.evaluate(pred_nb_acc)
f1_nb = evaluator_f1.evaluate(pred_nb_acc)

print("NB BEST by Accuracy -> TEST Accuracy:", acc_nb, "TEST F1:", f1_nb)
print("Best NB params (by Accuracy):", best_nb_acc_model.stages[-1].extractParamMap())

# --- best by F1 ---
cv_nb_f1 = CrossValidator(
    estimator=nb_pipeline,
    estimatorParamMaps=paramGrid_nb,
    evaluator=evaluator_f1,
    numFolds=3,
    seed=SEED,
    parallelism=1
)

cvModel_nb_f1 = cv_nb_f1.fit(train_df)
best_nb_f1_model = cvModel_nb_f1.bestModel

pred_nb_f1 = best_nb_f1_model.transform(test_df)
acc_nb2 = evaluator_acc.evaluate(pred_nb_f1)
f1_nb2 = evaluator_f1.evaluate(pred_nb_f1)

print("NB BEST by F1 -> TEST Accuracy:", acc_nb2, "TEST F1:", f1_nb2)
print("Best NB params (by F1):", best_nb_f1_model.stages[-1].extractParamMap())